# Results Compilation & Min-Max Normalization
Reads all 6 pkl files → builds 4 normalized tables → saves to Excel (4 sheets).

**Input pkl files:**
- `Strategy1_30runs_PartA.pkl` — S1 original 5 datasets (approach2 k)
- `Strategy1_30runs_PartB.pkl` — S1 new datasets (approach2 k)
- `S2A_results_kmf_30runs.pkl` / `S2A_results_ccf_30runs.pkl` — S2 PartA
- `S2B_results_kmf_30runs.pkl` / `S2B_results_ccf_30runs.pkl` — S2 PartB


## Cell 1: Imports

In [3]:
import pickle
import numpy as np
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
print("Imports OK")

Imports OK


## Cell 2: Load All pkl Files

In [5]:
print("Loading pkl files...")

with open('Strategy1_30runs_PartA.pkl', 'rb') as f: s1a = pickle.load(f)
with open('Strategy1_30runs_PartB.pkl', 'rb') as f: s1b = pickle.load(f)
with open('S2A_results_kmf_30runs.pkl', 'rb') as f: s2a_kmf = pickle.load(f)
with open('S2A_results_ccf_30runs-1.pkl', 'rb') as f: s2a_ccf = pickle.load(f)
with open('S2B_results_kmf_30runs-1.pkl', 'rb') as f: s2b_kmf = pickle.load(f)
with open('S2B_results_ccf_30runs.pkl', 'rb') as f: s2b_ccf = pickle.load(f)

print("All files loaded ✓")
print(f"S1A datasets: {s1a['datasets']}")
print(f"S1B datasets: {s1b['datasets']}")
print(f"S2A KMF datasets: {s2a_kmf['datasets']}")
print(f"S2B KMF datasets: {s2b_kmf['datasets']}")

Loading pkl files...
All files loaded ✓
S1A datasets: ['adult', 'compas', 'german', 'credit', 'law', 'adult_race', 'adult_combined', 'compas_race', 'compas_combined', 'law_race', 'law_combined']
S1B datasets: ['diabetes_gender', 'diabetes_race', 'diabetes_combined', 'dutch_gender', 'meps_gender', 'meps_race', 'meps_combined']
S2A KMF datasets: ['adult', 'compas', 'german', 'credit', 'law', 'adult_race', 'adult_combined', 'compas_race', 'compas_combined', 'law_race', 'law_combined']
S2B KMF datasets: ['diabetes_gender', 'diabetes_race', 'diabetes_combined', 'dutch_gender', 'meps_gender', 'meps_race', 'meps_combined']


## Cell 3: Configuration

In [7]:
METHOD_LABEL = {
    'kmeans':              'K-Means',
    'fairlet':             'Fairlet',
    'bfkm':               'BFKM',
    'fair_centroid':       'CCF',
    'postprocessing_nfp':  'PP-NFP',
    'postprocessing_gini': 'PP-Gini',
    'rawlsian':            'Rawlsian',
}
METHOD_ORDER = ['K-Means','Fairlet','BFKM','CCF','PP-NFP','PP-Gini','Rawlsian']

DS_LABEL = {
    'adult':'D1', 'compas':'D2', 'german':'D3', 'credit':'D4', 'law':'D5',
    'adult_race':'D1_race',     'adult_combined':'D1_comb',
    'compas_race':'D2_race',    'compas_combined':'D2_comb',
    'law_race':'D5_race',       'law_combined':'D5_comb',
    'diabetes_gender':'D6',     'diabetes_race':'D6_race', 'diabetes_combined':'D6_comb',
    'dutch_gender':'D7',
    'meps_gender':'D8',         'meps_race':'D8_race',     'meps_combined':'D8_comb',
}
DS_ORDER = list(DS_LABEL.keys())

UTIL_COLS = ['Inertia','Silhouette','Calinski_Harabasz','Davies_Bouldin','BCSS','Variance_Ratio']
FAIR_COLS = ['Balance','SPD','Disparate_Impact','Entropy']
HIGHER    = {'Silhouette','Calinski_Harabasz','BCSS','Variance_Ratio','Balance','Entropy'}
LOWER     = {'Inertia','Davies_Bouldin','SPD','Disparate_Impact'}

UTIL_DISPLAY = [
    ('SSE ↓',      'Inertia',           False),
    ('Silh. ↑',    'Silhouette',        True),
    ('CHI ↑',      'Calinski_Harabasz', True),
    ('DBI ↓',      'Davies_Bouldin',    False),
    ('BCSS ↑',     'BCSS',              True),
    ('V-Ratio ↑',  'Variance_Ratio',    True),
]
FAIR_DISPLAY = [
    ('Balance ↑',  'Balance',           True),
    ('SPD ↓',      'SPD',               False),
    ('|DI-1| ↓',   'Disparate_Impact',  False),
    ('Entropy ↑',  'Entropy',           True),
]
print("Config loaded ✓")

Config loaded ✓


## Cell 4: Extract from pkl → flat DataFrames

In [9]:
def extract_s1(pkl):
    """
    S1 structure (new): pkl['clustering_results_30'][ds][method]
    S1 structure (old): pkl['clustering_results_30']['approach2'][ds][method]
    """
    results = pkl['clustering_results_30']
    first_key = list(results.keys())[0]
    if first_key in ['approach1', 'approach2']:
        results = results.get('approach2', results.get('approach1', {}))
    rows_q, rows_f = [], []
    for ds, methods in results.items():
        for method, r in methods.items():
            base = {
                'Dataset': ds,
                'Method':  METHOD_LABEL.get(method, method),
                'k':       r['k'],
                'N_runs':  r['n_runs']
            }
            rows_q.append({**base,
                **{c+'_mean': r['mean_quality'].get(c, np.nan) for c in UTIL_COLS},
                **{c+'_std':  r['std_quality'].get(c,  np.nan) for c in UTIL_COLS}})
            rows_f.append({**base,
                **{c+'_mean': r['mean_fairness'].get(c, np.nan) for c in FAIR_COLS},
                **{c+'_std':  r['std_fairness'].get(c,  np.nan) for c in FAIR_COLS}})
    return pd.DataFrame(rows_q), pd.DataFrame(rows_f)

def extract_s2(pkl):
    """S2 structure: pkl['final_results_30'][ds][method]"""
    results = pkl['final_results_30']
    rows_q, rows_f = [], []
    for ds, methods in results.items():
        for method, r in methods.items():
            base = {
                'Dataset': ds,
                'Method':  METHOD_LABEL.get(method.lower(), method),
                'k':       r['k'],
                'N_runs':  r['n_runs']
            }
            rows_q.append({**base,
                **{c+'_mean': r['mean_quality'].get(c, np.nan) for c in UTIL_COLS},
                **{c+'_std':  r['std_quality'].get(c,  np.nan) for c in UTIL_COLS}})
            rows_f.append({**base,
                **{c+'_mean': r['mean_fairness'].get(c, np.nan) for c in FAIR_COLS},
                **{c+'_std':  r['std_fairness'].get(c,  np.nan) for c in FAIR_COLS}})
    return pd.DataFrame(rows_q), pd.DataFrame(rows_f)

# ── Strategy 1: merge PartA + PartB ──────────────────────────────────────────
q1a, f1a = extract_s1(s1a)
q1b, f1b = extract_s1(s1b)
q1 = pd.concat([q1a, q1b], ignore_index=True)
f1 = pd.concat([f1a, f1b], ignore_index=True)

# ── Strategy 2: merge all 4 pkls ─────────────────────────────────────────────
# Must concat then deduplicate by (Dataset, Method) — same dataset appears in
# both kmf and ccf pkls with different methods, so concat is correct here.
q2_parts, f2_parts = [], []
for pkl in [s2a_kmf, s2a_ccf, s2b_kmf, s2b_ccf]:
    q, f = extract_s2(pkl)
    q2_parts.append(q); f2_parts.append(f)
q2 = pd.concat(q2_parts, ignore_index=True)
f2 = pd.concat(f2_parts, ignore_index=True)

# Remove duplicates if any (keep first)
q2 = q2.drop_duplicates(subset=['Dataset','Method'], keep='first')
f2 = f2.drop_duplicates(subset=['Dataset','Method'], keep='first')

print(f"S1 Quality: {q1.shape}  S1 Fairness: {f1.shape}")
print(f"S2 Quality: {q2.shape}  S2 Fairness: {f2.shape}")
print(f"S1 datasets: {sorted(q1['Dataset'].unique())}")
print(f"S2 datasets: {sorted(q2['Dataset'].unique())}")
print(f"S2 methods per dataset check:")
print(q2.groupby('Dataset')['Method'].count())


S1 Quality: (126, 16)  S1 Fairness: (126, 12)
S2 Quality: (126, 16)  S2 Fairness: (126, 12)
S1 datasets: ['adult', 'adult_combined', 'adult_race', 'compas', 'compas_combined', 'compas_race', 'credit', 'diabetes_combined', 'diabetes_gender', 'diabetes_race', 'dutch_gender', 'german', 'law', 'law_combined', 'law_race', 'meps_combined', 'meps_gender', 'meps_race']
S2 datasets: ['adult', 'adult_combined', 'adult_race', 'compas', 'compas_combined', 'compas_race', 'credit', 'diabetes_combined', 'diabetes_gender', 'diabetes_race', 'dutch_gender', 'german', 'law', 'law_combined', 'law_race', 'meps_combined', 'meps_gender', 'meps_race']
S2 methods per dataset check:
Dataset
adult                7
adult_combined       7
adult_race           7
compas               7
compas_combined      7
compas_race          7
credit               7
diabetes_combined    7
diabetes_gender      7
diabetes_race        7
dutch_gender         7
german               7
law                  7
law_combined         7
law_

## Cell 5: Min-Max Normalization [0,1]

In [11]:
def minmax_norm_per_row(df, cols):
    """
    Per-row: for each (Dataset) group within each metric column,
    normalize the 7 method values to [0,1].
    Since df is long format (one row per Dataset×Method),
    group by Dataset and normalize mean values across methods.
    """
    df = df.copy()
    for col in cols:
        mean_col = col + '_mean'
        std_col  = col + '_std'
        if mean_col not in df.columns: continue

        # For each dataset, normalize across all 7 methods
        for ds, group_idx in df.groupby('Dataset').groups.items():
            vals = df.loc[group_idx, mean_col]
            mn   = vals.min()
            mx   = vals.max()
            rng  = mx - mn
            if rng > 0:
                df.loc[group_idx, mean_col] = ((vals - mn) / rng).round(3)
                if std_col in df.columns:
                    df.loc[group_idx, std_col] = (df.loc[group_idx, std_col] / rng).round(3)
            else:
                df.loc[group_idx, mean_col] = 0.5
                if std_col in df.columns:
                    df.loc[group_idx, std_col] = 0.0
    return df

q1_n = minmax_norm_per_row(q1, UTIL_COLS)
f1_n = minmax_norm_per_row(f1, FAIR_COLS)
q2_n = minmax_norm_per_row(q2, UTIL_COLS)
f2_n = minmax_norm_per_row(f2, FAIR_COLS)

print("Per-row normalization complete ✓")
print("\nVerification — D1 SSE (should span 0 to 1):")
sample = q1_n[q1_n['Dataset']=='adult'][['Method','Inertia_mean']].sort_values('Inertia_mean')
print(sample.to_string(index=False))
print(f"Min={sample['Inertia_mean'].min()}, Max={sample['Inertia_mean'].max()}")

Per-row normalization complete ✓

Verification — D1 SSE (should span 0 to 1):
  Method  Inertia_mean
 K-Means         0.000
  PP-NFP         0.024
 PP-Gini         0.025
     CCF         0.086
    BFKM         0.156
Rawlsian         0.225
 Fairlet         1.000
Min=0.0, Max=1.0


## Cell 6: Build Pivot Tables

In [13]:
def build_pivot(df, display_metrics, fmt='combined', show_k_per_method=False):
    """
    fmt='combined'         → "0.733±0.002"
    show_k_per_method=True → "0.733±0.002(5)" — show each method's k value
    Tied best: ALL methods achieving best value are marked bold (not just one)
    """
    ds_present = [d for d in DS_ORDER if d in df["Dataset"].unique()]
    rows = []
    for label, col, higher in display_metrics:
        mean_col = col + "_mean"
        std_col  = col + "_std"
        if mean_col not in df.columns:
            continue
        for ds in ds_present:
            sub = df[df["Dataset"] == ds]
            if sub.empty:
                continue
            k_val  = int(sub["k"].mode()[0])
            n_runs = int(sub["N_runs"].mode()[0])

            # Collect all mean values
            num_vals = {}
            for m in METHOD_ORDER:
                ms = sub[sub["Method"] == m]
                if len(ms) > 0:
                    v = ms[mean_col].iloc[0]
                    if not np.isnan(v):
                        num_vals[m] = v

            # Find best value (may be tied)
            if num_vals:
                best_val = max(num_vals.values()) if higher else min(num_vals.values())
                # All methods within 1e-6 of best are co-best
                best_methods = {m for m, v in num_vals.items()
                                if abs(v - best_val) < 1e-6}
            else:
                best_methods = set()

            row = {
                "Metric":  label,
                "Dataset": DS_LABEL.get(ds, ds),
                "k":       "" if show_k_per_method else k_val,
                "_col":    col,
                "_higher": higher,
                "_best":   best_methods,   # now a SET of best methods
                "_type":   "mean",
            }

            for m in METHOD_ORDER:
                ms = sub[sub["Method"] == m]
                if len(ms) > 0:
                    mean_v = ms[mean_col].iloc[0]
                    std_v  = ms[std_col].iloc[0] if std_col in ms.columns else 0.0
                    k_m    = int(ms["k"].iloc[0])
                    if show_k_per_method:
                        row[m] = f"{mean_v:.3f}\u00b1{std_v:.3f}(k={k_m})"
                    else:
                        row[m] = f"{mean_v:.3f}\u00b1{std_v:.3f}"
                    row[f"_{m}_mean"] = mean_v
                    row[f"_{m}_k"]    = k_m
                else:
                    row[m]            = ""
                    row[f"_{m}_mean"] = np.nan
                    row[f"_{m}_k"]    = np.nan
            rows.append(row)
    return rows

# Strategy 1: combined format, shared k per dataset
p_s1q = build_pivot(q1_n, UTIL_DISPLAY, fmt="combined", show_k_per_method=False)
p_s1f = build_pivot(f1_n, FAIR_DISPLAY, fmt="combined", show_k_per_method=False)

# Strategy 2: show k per method (each method has its own majority vote k)
p_s2q = build_pivot(q2_n, UTIL_DISPLAY, fmt="combined", show_k_per_method=True)
p_s2f = build_pivot(f2_n, FAIR_DISPLAY, fmt="combined", show_k_per_method=True)

# Preview
preview_cols = ["Metric","Dataset","k"] + METHOD_ORDER
print("=== S1 Utility (preview) ===")
print(pd.DataFrame(p_s1q[:6])[preview_cols].to_string(index=False))
print("\n=== S2 Utility (preview, with k) ===")
print(pd.DataFrame(p_s2q[:3])[preview_cols].to_string(index=False))


=== S1 Utility (preview) ===
Metric Dataset  k     K-Means     Fairlet        BFKM         CCF      PP-NFP     PP-Gini    Rawlsian
 SSE ↓      D1  5 0.000±0.046 1.000±0.012 0.156±0.092 0.086±0.053 0.024±0.044 0.025±0.044 0.225±0.048
 SSE ↓      D2  2 0.000±0.000 1.000±0.007 0.002±0.000 0.057±0.000 0.001±0.000 0.006±0.000 0.000±0.000
 SSE ↓      D3  2 0.000±0.000 1.000±0.039 0.077±0.159 0.041±0.007 0.021±0.001 0.071±0.000 0.000±0.000
 SSE ↓      D4  2 0.000±0.000 1.000±0.007 0.000±0.000 0.049±0.001 0.001±0.000 0.003±0.000 0.000±0.000
 SSE ↓      D5  4 0.000±0.026 1.000±0.005 0.075±0.052 0.049±0.018 0.004±0.026 0.006±0.026 0.079±0.000
 SSE ↓ D1_race  5 0.000±0.039 1.000±0.004 0.130±0.077 0.062±0.047 0.002±0.039 0.005±0.039 0.172±0.026

=== S2 Utility (preview, with k) ===
Metric Dataset k          K-Means          Fairlet             BFKM              CCF           PP-NFP          PP-Gini         Rawlsian
 SSE ↓      D1   0.000±0.005(k=8) 1.000±0.005(k=4) 0.153±0.070(k=8) 0.909±0.000(k=2

## Cell 7: Write Excel

In [15]:
thin   = Side(style="thin", color="CCCCCC")
bdr    = Border(left=thin, right=thin, top=thin, bottom=thin)
H_FILL = PatternFill("solid", start_color="2E74B5")
M_FILL = PatternFill("solid", start_color="D6E4F0")
A_FILL = PatternFill("solid", start_color="F5F9FD")
B_FILL = PatternFill("solid", start_color="FFF2CC")

def write_sheet(ws, rows):
    display_cols = ["Metric", "Dataset", "k"] + METHOD_ORDER

    # Header
    for ci, h in enumerate(display_cols, 1):
        c = ws.cell(1, ci, h)
        c.font      = Font(name="Arial", bold=True, color="FFFFFF", size=10)
        c.fill      = H_FILL
        c.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        c.border    = bdr
    ws.row_dimensions[1].height = 28

    cur_metric, metric_row_start = None, 2
    er = 2

    for row in rows:
        m           = row["Metric"]
        higher      = row["_higher"]
        best_set    = row["_best"]   # SET of best methods

        # Metric merge tracking
        if m != cur_metric and m != "":
            if cur_metric and er - metric_row_start > 1:
                ws.merge_cells(start_row=metric_row_start, start_column=1,
                               end_row=er-1, end_column=1)
                ws.cell(metric_row_start, 1).alignment = Alignment(
                    horizontal="center", vertical="center", wrap_text=True)
            cur_metric, metric_row_start = m, er

        alt    = (er % 2 == 0)
        row_bg = A_FILL if alt else None

        # Metric, Dataset, k
        for ci, key in enumerate(["Metric", "Dataset", "k"], 1):
            c = ws.cell(er, ci, row.get(key, ""))
            c.font      = Font(name="Arial", size=9,
                               bold=(ci==1), color="1F4E79" if ci==1 else "000000")
            c.fill      = M_FILL if ci==1 else (row_bg or PatternFill())
            c.alignment = Alignment(horizontal="center", vertical="center")
            c.border    = bdr

        # Method value columns — bold if in best_set (tied best allowed)
        for ci, mth in enumerate(METHOD_ORDER, 4):
            val     = row.get(mth, "")
            is_best = (mth in best_set)
            # Add line breaks for wrapped display in Excel
            if isinstance(val, str):
                val = val.replace("±", "\n±").replace("(k=", "\n(k=")
            c = ws.cell(er, ci, val)
            c.font      = Font(name="Arial", size=9, bold=is_best)
            c.fill      = B_FILL if is_best else (row_bg or PatternFill())
            c.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
            c.border    = bdr

        ws.row_dimensions[er].height = 28
        er += 1

    # Merge last metric group
    if cur_metric and er - metric_row_start > 1:
        ws.merge_cells(start_row=metric_row_start, start_column=1,
                       end_row=er-1, end_column=1)
        ws.cell(metric_row_start, 1).alignment = Alignment(
            horizontal="center", vertical="center", wrap_text=True)

    # Column widths — wider for S2 (has k info in cell)
    ws.column_dimensions["A"].width = 12
    ws.column_dimensions["B"].width = 14
    ws.column_dimensions["C"].width = 6
    for ci in range(4, len(METHOD_ORDER)+5):
        ws.column_dimensions[get_column_letter(ci)].width = 18
    ws.freeze_panes = "A2"

# Build workbook
wb = Workbook()
wb.remove(wb.active)

for title, rows in [("S1 Utility",  p_s1q),
                    ("S1 Fairness", p_s1f),
                    ("S2 Utility",  p_s2q),
                    ("S2 Fairness", p_s2f)]:
    ws = wb.create_sheet(title)
    write_sheet(ws, rows)
    print(f"✓ Sheet \"{title}\": {len(rows)} rows")

wb.save("Results_Tables_30runs.xlsx")
print("\n✓ Saved: Results_Tables_30runs.xlsx")
print("  S1: mean±std (shared k per dataset, best values bold+yellow)")
print("  S2: mean±std(k=N) per method (each method\'s majority vote k shown)")
print("  Tied best values: ALL tied methods are bold+yellow")


✓ Sheet "S1 Utility": 108 rows
✓ Sheet "S1 Fairness": 72 rows
✓ Sheet "S2 Utility": 108 rows
✓ Sheet "S2 Fairness": 72 rows

✓ Saved: Results_Tables_30runs.xlsx
  S1: mean±std (shared k per dataset, best values bold+yellow)
  S2: mean±std(k=N) per method (each method's majority vote k shown)
  Tied best values: ALL tied methods are bold+yellow


In [16]:
# =============================================================================
# Export Raw (Unscaled) Data to Excel
# Same format as normalized tables but using original values
# =============================================================================

from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

thin   = Side(style='thin', color='CCCCCC')
bdr    = Border(left=thin, right=thin, top=thin, bottom=thin)
H_FILL = PatternFill('solid', start_color='2E74B5')
M_FILL = PatternFill('solid', start_color='D6E4F0')
A_FILL = PatternFill('solid', start_color='F5F9FD')
B_FILL = PatternFill('solid', start_color='FFF2CC')
NO_FILL= PatternFill(fill_type=None)

def build_pivot_raw(df, display_metrics, show_k_per_method=False):
    """Same as build_pivot but uses original (non-normalized) values."""
    ds_present = [d for d in DS_ORDER if d in df["Dataset"].unique()]
    rows = []
    for label, col, higher in display_metrics:
        mean_col = col + "_mean"
        std_col  = col + "_std"
        if mean_col not in df.columns: continue
        for ds in ds_present:
            sub = df[df["Dataset"] == ds]
            if sub.empty: continue
            k_val = int(sub["k"].mode()[0])

            # Find best value(s)
            num_vals = {}
            for m in METHOD_ORDER:
                ms = sub[sub["Method"] == m]
                if len(ms) > 0:
                    v = ms[mean_col].iloc[0]
                    if not np.isnan(v):
                        num_vals[m] = v
            if num_vals:
                best_val = max(num_vals.values()) if higher else min(num_vals.values())
                best_set = {m for m, v in num_vals.items() if abs(v - best_val) < 1e-9}
            else:
                best_set = set()

            row = {
                "Metric":  label,
                "Dataset": DS_LABEL.get(ds, ds),
                "k":       k_val if not show_k_per_method else "-",
                "_col":    col,
                "_higher": higher,
                "_best":   best_set,
            }
            for m in METHOD_ORDER:
                ms = sub[sub["Method"] == m]
                if len(ms) > 0:
                    mean_v = ms[mean_col].iloc[0]
                    std_v  = ms[std_col].iloc[0] if std_col in ms.columns else 0.0
                    k_m    = int(ms["k"].iloc[0])
                    if show_k_per_method:
                        val = f"{mean_v:.4f}\n±{std_v:.4f}\n(k={k_m})"
                    else:
                        val = f"{mean_v:.4f}\n±{std_v:.4f}"
                    row[m]            = val
                    row[f"_{m}_mean"] = mean_v
                else:
                    row[m]            = ""
                    row[f"_{m}_mean"] = np.nan
            rows.append(row)
    return rows

def write_raw_sheet(ws, rows, col_width=16):
    display_cols = ["Metric", "Dataset", "k"] + METHOD_ORDER

    # Header
    for ci, h in enumerate(display_cols, 1):
        c = ws.cell(1, ci, h)
        c.font      = Font(name="Arial", bold=True, color="FFFFFF", size=10)
        c.fill      = H_FILL
        c.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        c.border    = bdr
    ws.row_dimensions[1].height = 28

    cur_metric, metric_row_start = None, 2
    er = 2

    for row in rows:
        m        = row["Metric"]
        best_set = row["_best"]

        if m != cur_metric and m != "":
            if cur_metric and er - metric_row_start > 1:
                ws.merge_cells(start_row=metric_row_start, start_column=1,
                               end_row=er-1, end_column=1)
                ws.cell(metric_row_start, 1).alignment = Alignment(
                    horizontal="center", vertical="center", wrap_text=True)
            cur_metric, metric_row_start = m, er

        alt    = (er % 2 == 0)
        row_bg = A_FILL if alt else NO_FILL

        # Metric, Dataset, k
        for ci, key in enumerate(["Metric", "Dataset", "k"], 1):
            c = ws.cell(er, ci, row.get(key, ""))
            c.font      = Font(name="Arial", size=9,
                               bold=(ci==1), color="1F4E79" if ci==1 else "000000")
            c.fill      = M_FILL if ci==1 else (row_bg or NO_FILL)
            c.alignment = Alignment(horizontal="center", vertical="center")
            c.border    = bdr

        # Method values
        for ci, mth in enumerate(METHOD_ORDER, 4):
            val     = row.get(mth, "")
            is_best = (mth in best_set)
            c = ws.cell(er, ci, val)
            c.font      = Font(name="Arial", size=9, bold=is_best)
            c.fill      = B_FILL if is_best else (row_bg or NO_FILL)
            c.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
            c.border    = bdr

        ws.row_dimensions[er].height = 36
        er += 1

    # Merge last metric group
    if cur_metric and er - metric_row_start > 1:
        ws.merge_cells(start_row=metric_row_start, start_column=1,
                       end_row=er-1, end_column=1)
        ws.cell(metric_row_start, 1).alignment = Alignment(
            horizontal="center", vertical="center", wrap_text=True)

    ws.column_dimensions["A"].width = 12
    ws.column_dimensions["B"].width = 14
    ws.column_dimensions["C"].width = 6
    for ci in range(4, len(METHOD_ORDER)+5):
        ws.column_dimensions[get_column_letter(ci)].width = col_width
    ws.freeze_panes = "A2"

# Build raw pivot tables (use q1, f1, q2, f2 — NOT normalized versions)
p_raw_s1q = build_pivot_raw(q1, UTIL_DISPLAY, show_k_per_method=False)
p_raw_s1f = build_pivot_raw(f1, FAIR_DISPLAY, show_k_per_method=False)
p_raw_s2q = build_pivot_raw(q2, UTIL_DISPLAY, show_k_per_method=True)
p_raw_s2f = build_pivot_raw(f2, FAIR_DISPLAY, show_k_per_method=True)

# Write Excel
wb_raw = Workbook()
wb_raw.remove(wb_raw.active)

for title, rows, cw in [
    ("S1 Utility (Raw)",  p_raw_s1q, 14),
    ("S1 Fairness (Raw)", p_raw_s1f, 14),
    ("S2 Utility (Raw)",  p_raw_s2q, 18),
    ("S2 Fairness (Raw)", p_raw_s2f, 18),
]:
    ws = wb_raw.create_sheet(title)
    write_raw_sheet(ws, rows, col_width=cw)
    print(f"✓ Sheet \"{title}\": {len(rows)} rows")

wb_raw.save("Results_Tables_Raw.xlsx")
print("\n✓ Saved: Results_Tables_Raw.xlsx")
print("  Format: mean±std (4 decimal places), raw unscaled values")
print("  S1: shared k per dataset")
print("  S2: per-method k in parentheses")
print("  Best values: yellow highlight + bold (ties included)")


✓ Sheet "S1 Utility (Raw)": 108 rows
✓ Sheet "S1 Fairness (Raw)": 72 rows
✓ Sheet "S2 Utility (Raw)": 108 rows
✓ Sheet "S2 Fairness (Raw)": 72 rows

✓ Saved: Results_Tables_Raw.xlsx
  Format: mean±std (4 decimal places), raw unscaled values
  S1: shared k per dataset
  S2: per-method k in parentheses
  Best values: yellow highlight + bold (ties included)
